# 🌐 TranslateGemma:27B Translation Notebook

**High-Quality Translation with Google's TranslateGemma Model via Ollama**

This notebook is specially curated for `translategemma:27b` from Ollama - Google's powerful open translation model built on Gemma 3.

**Features:**
- 📤 Upload any text file to translate
- 🌍 Supports **55 languages** (production-tier quality)
- 🦙 Uses Ollama locally in Colab
- 📥 Download translated output

**Supported Languages Include:**
Hindi, Bengali, Tamil, Telugu, Spanish, French, German, Chinese, Japanese, Korean, Arabic, Russian, Portuguese, and 42 more!

---
⚠️ **Note:** This notebook requires GPU runtime for optimal performance.
Go to **Runtime → Change runtime type → GPU** (T4 recommended)

## 📦 Step 1: Install Dependencies
Run this cell to install all required packages.

In [ ]:
# Install required packages
!pip install -q ollama colorama

print("✅ Dependencies installed!")

## 🦙 Step 2: Install & Start Ollama Server
This cell installs Ollama and starts the server in the background.

In [ ]:
# Install and start Ollama server
import subprocess
import time
import os

print("🦙 Installing Ollama...")

# Install zstd first (required for Ollama extraction)
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1

# Download and install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

print("\n🚀 Starting Ollama server in background...")

# Start Ollama server in background
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
subprocess.Popen(['/usr/local/bin/ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for server to start
time.sleep(5)

# Verify server is running
try:
    import ollama
    ollama.list()
    print("✅ Ollama server is running!")
except Exception as e:
    print(f"⚠️ Ollama server may not be ready yet. Error: {e}")
    print("   Please wait a few seconds and run this cell again.")

## 📥 Step 3: Download TranslateGemma:27B Model
This will download the translategemma:27b model (~16GB). This may take several minutes.

In [ ]:
# Pull translategemma:27b model
import ollama

MODEL_NAME = "translategemma:27b"

print(f"📥 Pulling model: {MODEL_NAME}")
print("   This may take several minutes (~16GB download)...\n")

try:
    # Pull with progress
    current_digest = ''
    for progress in ollama.pull(MODEL_NAME, stream=True):
        digest = progress.get('digest', '')
        if digest != current_digest and current_digest:
            print()  # Newline between layers
        current_digest = digest

        status = progress.get('status', '')
        if 'completed' in progress and 'total' in progress:
            completed = progress['completed']
            total = progress['total']
            pct = (completed / total * 100) if total > 0 else 0
            print(f"\r   {status}: {pct:.1f}% ({completed}/{total})", end='', flush=True)
        else:
            print(f"\r   {status}", end='', flush=True)

    print(f"\n\n✅ Model '{MODEL_NAME}' downloaded successfully!")

except Exception as e:
    print(f"\n❌ Error pulling model: {e}")
    print("   Make sure Ollama server is running (run the previous cell first).")

## 📤 Step 4: Upload Your Text File
Upload the text file you want to translate.

In [ ]:
from google.colab import files
import os

print("📤 Please upload your text file to translate:")
uploaded = files.upload()

# Get the uploaded file name
UPLOADED_FILE = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {UPLOADED_FILE}")
print(f"📄 File size: {len(uploaded[UPLOADED_FILE])} bytes")

# Display preview
with open(UPLOADED_FILE, 'r', encoding='utf-8') as f:
    content = f.read()
    word_count = len(content.split())
    char_count = len(content)

print(f"\n📊 Content stats:")
print(f"   Words: {word_count:,}")
print(f"   Characters: {char_count:,}")
print(f"\n📝 Preview (first 500 chars):\n{content[:500]}...")

## 🌍 Step 5: Select Target Language
Choose from the 55 languages supported by TranslateGemma:27B.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

# TranslateGemma:27B Supported Languages (55 production-tier languages)
TRANSLATEGEMMA_LANGUAGES = {
    # Major Indian Languages
    "Hindi (हिन्दी)": ("hi", "Hindi"),
    "Bengali (বাংলা)": ("bn", "Bengali"),
    "Tamil (தமிழ்)": ("ta", "Tamil"),
    "Telugu (తెలుగు)": ("te", "Telugu"),
    "Marathi (मराठी)": ("mr", "Marathi"),
    "Gujarati (ગુજરાતી)": ("gu", "Gujarati"),
    "Kannada (ಕನ್ನಡ)": ("kn", "Kannada"),
    "Malayalam (മലയാളം)": ("ml", "Malayalam"),
    "Nepali (नेपाली)": ("ne", "Nepali"),
    
    # European Languages
    "Spanish (Español)": ("es", "Spanish"),
    "French (Français)": ("fr", "French"),
    "German (Deutsch)": ("de", "German"),
    "Italian (Italiano)": ("it", "Italian"),
    "Portuguese (Português)": ("pt", "Portuguese"),
    "Dutch (Nederlands)": ("nl", "Dutch"),
    "Polish (Polski)": ("pl", "Polish"),
    "Romanian (Română)": ("ro", "Romanian"),
    "Greek (Ελληνικά)": ("el", "Greek"),
    "Czech (Čeština)": ("cs", "Czech"),
    "Slovak (Slovenčina)": ("sk", "Slovak"),
    "Hungarian (Magyar)": ("hu", "Hungarian"),
    "Bulgarian (Български)": ("bg", "Bulgarian"),
    "Croatian (Hrvatski)": ("hr", "Croatian"),
    "Serbian (Српски)": ("sr", "Serbian"),
    "Slovenian (Slovenščina)": ("sl", "Slovenian"),
    "Macedonian (Македонски)": ("mk", "Macedonian"),
    "Albanian (Shqip)": ("sq", "Albanian"),
    "Icelandic (Íslenska)": ("is", "Icelandic"),
    "Norwegian (Norsk)": ("no", "Norwegian"),
    "Swedish (Svenska)": ("sv", "Swedish"),
    "Danish (Dansk)": ("da", "Danish"),
    "Finnish (Suomi)": ("fi", "Finnish"),
    "Estonian (Eesti)": ("et", "Estonian"),
    "Latvian (Latviešu)": ("lv", "Latvian"),
    "Lithuanian (Lietuvių)": ("lt", "Lithuanian"),
    "Catalan (Català)": ("ca", "Catalan"),
    
    # Slavic & Eastern European
    "Russian (Русский)": ("ru", "Russian"),
    "Ukrainian (Українська)": ("uk", "Ukrainian"),
    "Belarusian (Беларуская)": ("be", "Belarusian"),
    
    # Asian Languages
    "Chinese Simplified (简体中文)": ("zh-Hans", "Chinese"),
    "Japanese (日本語)": ("ja", "Japanese"),
    "Korean (한국어)": ("ko", "Korean"),
    "Thai (ไทย)": ("th", "Thai"),
    "Vietnamese (Tiếng Việt)": ("vi", "Vietnamese"),
    "Indonesian (Bahasa Indonesia)": ("id", "Indonesian"),
    "Filipino (Tagalog)": ("fil", "Filipino"),
    
    # Middle Eastern & Central Asian
    "Arabic (العربية)": ("ar", "Arabic"),
    "Persian (فارسی)": ("fa", "Persian"),
    "Hebrew (עברית)": ("he", "Hebrew"),
    "Turkish (Türkçe)": ("tr", "Turkish"),
    "Georgian (ქართული)": ("ka", "Georgian"),
    "Azerbaijani (Azərbaycan)": ("az", "Azerbaijani"),
    "Kazakh (Қазақ)": ("kk", "Kazakh"),
    "Mongolian (Монгол)": ("mn", "Mongolian"),
    
    # African Languages
    "Swahili (Kiswahili)": ("sw", "Swahili"),
    "Amharic (አማርኛ)": ("am", "Amharic"),
    "Afrikaans": ("af", "Afrikaans"),
}

# Source language (default English)
SOURCE_LANG_CODE = "en"
SOURCE_LANG_NAME = "English"

display(HTML("<h3>🌍 Select Target Language</h3>"))
display(HTML("<p>TranslateGemma:27B supports <strong>55 languages</strong> with production-tier quality.</p>"))

# Language dropdown
language_dropdown = widgets.Dropdown(
    options=list(TRANSLATEGEMMA_LANGUAGES.keys()),
    value="Hindi (हिन्दी)",
    description='Translate to:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Chunk size slider
chunk_size_slider = widgets.IntSlider(
    value=300,
    min=100,
    max=800,
    step=50,
    description='Chunk Size (words):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

display(language_dropdown)
display(HTML("<br>"))
display(chunk_size_slider)

print("\n💡 Tip: Smaller chunk sizes (200-300) may produce better quality for complex texts.")

In [ ]:
# Save configuration
selected_lang = TRANSLATEGEMMA_LANGUAGES[language_dropdown.value]
TARGET_LANG_CODE = selected_lang[0]
TARGET_LANG_NAME = selected_lang[1]
CHUNK_SIZE = chunk_size_slider.value

print(f"\n✅ Configuration saved:")
print(f"   🦙 Model: translategemma:27b")
print(f"   📖 Source: English (en)")
print(f"   🌐 Target: {TARGET_LANG_NAME} ({TARGET_LANG_CODE})")
print(f"   📦 Chunk Size: {CHUNK_SIZE} words")

## 🚀 Step 6: Run Translation
This cell contains the translation engine and will translate your file.

In [ ]:
#!/usr/bin/env python3
"""
TranslateGemma:27B Translation Engine
Optimized for the translategemma:27b model from Ollama
"""

import os
import re
import time
from datetime import datetime
from pathlib import Path
import ollama


def chunk_text(text, chunk_words=300):
    """Split text into chunks at paragraph boundaries."""
    paragraph_patterns = [
        r'\n\s*\n',
        r'\r\n\s*\r\n',
        r'\n\s{2,}\n',
    ]
    paragraph_split_pattern = '|'.join(paragraph_patterns)
    paragraphs = re.split(paragraph_split_pattern, text)
    paragraphs = [para.strip() for para in paragraphs if para.strip()]

    chunks = []
    current_chunk = []
    current_count = 0

    for para in paragraphs:
        para_words = para.split()
        para_count = len(para_words)

        if para_count > chunk_words:
            if current_chunk:
                chunks.append('\n\n'.join(current_chunk))
                current_chunk = []
                current_count = 0

            words = para.split()
            for i in range(0, len(words), chunk_words):
                chunk_words_list = words[i:i + chunk_words]
                chunk_text = ' '.join(chunk_words_list)
                chunks.append(chunk_text)
        else:
            if current_count + para_count > chunk_words and current_chunk:
                chunks.append('\n\n'.join(current_chunk))
                current_chunk = [para]
                current_count = para_count
            else:
                current_chunk.append(para)
                current_count += para_count

    if current_chunk:
        chunks.append('\n\n'.join(current_chunk))

    return chunks


def clean_translation(text):
    """Clean up translation artifacts."""
    # Remove thinking tags
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    # Remove code blocks
    text = re.sub(r'```\w*\n?', '', text)
    # Clean up lines
    lines = [line.strip() for line in text.split('\n')]
    text = '\n\n'.join(line for line in lines if line)
    return text.strip()


def build_translategemma_prompt(source_lang, source_code, target_lang, target_code, text):
    """
    Build the official TranslateGemma prompt format.
    
    Format: You are a professional {SOURCE_LANG} ({SOURCE_CODE}) to {TARGET_LANG} ({TARGET_CODE}) translator.
    Your goal is to accurately convey the meaning and nuances of the original {SOURCE_LANG} text
    while adhering to {TARGET_LANG} grammar, vocabulary, and cultural sensitivities.
    Produce only the {TARGET_LANG} translation, without any additional explanations or commentary.
    Please translate the following {SOURCE_LANG} text into {TARGET_LANG}: {TEXT}
    """
    prompt = f"""You are a professional {source_lang} ({source_code}) to {target_lang} ({target_code}) translator. Your goal is to accurately convey the meaning and nuances of the original {source_lang} text while adhering to {target_lang} grammar, vocabulary, and cultural sensitivities. Produce only the {target_lang} translation, without any additional explanations or commentary. Please translate the following {source_lang} text into {target_lang}: {text}"""
    return prompt


class TranslateGemmaEngine:
    """Translation engine using TranslateGemma:27B via Ollama."""

    def __init__(self, source_lang="English", source_code="en", target_lang="Hindi", target_code="hi"):
        self.model_name = "translategemma:27b"
        self.source_lang = source_lang
        self.source_code = source_code
        self.target_lang = target_lang
        self.target_code = target_code
        
        print(f"📥 Initializing TranslateGemma:27B engine")
        print(f"   Source: {source_lang} ({source_code})")
        print(f"   Target: {target_lang} ({target_code})")

        # Verify model is available
        try:
            models = ollama.list()
            available = [m.get('name', '') for m in models.get('models', [])]
            
            if not any('translategemma' in m.lower() for m in available):
                print(f"⚠️ Model '{self.model_name}' not found. Attempting to pull...")
                ollama.pull(self.model_name)
                print(f"✅ Model pulled successfully!")
            else:
                print(f"✅ Model is available!")
        except Exception as e:
            print(f"❌ Error initializing: {e}")
            raise

    def translate(self, text):
        """Translate text using TranslateGemma's official prompt format."""
        prompt = build_translategemma_prompt(
            self.source_lang, self.source_code,
            self.target_lang, self.target_code,
            text
        )

        try:
            response = ollama.generate(
                model=self.model_name,
                prompt=prompt,
                options={
                    "temperature": 0.3,
                    "num_predict": 2048,
                }
            )

            translation = response.get('response', '')
            return clean_translation(translation)

        except Exception as e:
            print(f"❌ Translation error: {e}")
            raise


def translate_file(input_file, source_lang, source_code, target_lang, target_code, chunk_size=300):
    """Translate entire file using TranslateGemma:27B."""
    
    print(f"\n{'=' * 70}")
    print(f"🌐 TRANSLATEGEMMA:27B TRANSLATOR")
    print(f"{'=' * 70}")
    print(f"📄 Input: {input_file}")
    print(f"🦙 Model: translategemma:27b")
    print(f"📖 Source: {source_lang} ({source_code})")
    print(f"🌐 Target: {target_lang} ({target_code})")
    print(f"{'=' * 70}\n")

    # Initialize engine
    engine = TranslateGemmaEngine(source_lang, source_code, target_lang, target_code)

    # Read input
    with open(input_file, 'r', encoding='utf-8') as f:
        text = f.read()

    # Clean markers
    lines = text.split('\n')
    cleaned = [l for l in lines if not (l.strip().startswith('===') and l.strip().endswith('==='))]
    text = '\n'.join(cleaned).strip()

    orig_words = len(text.split())
    orig_chars = len(text)
    print(f"📊 Input: {orig_chars:,} chars, {orig_words:,} words")

    # Chunk text
    print(f"\n📦 Creating chunks ({chunk_size} words each)...")
    chunks = chunk_text(text, chunk_size)
    print(f"✅ Created {len(chunks)} chunks")

    # Translate chunks
    print(f"\n🎯 STARTING TRANSLATION\n")

    translations = []
    start_time = time.time()

    for i, chunk in enumerate(chunks, 1):
        chunk_start = time.time()

        print(f"\n{'=' * 50}")
        print(f"📄 Chunk {i}/{len(chunks)}")
        print(f"   Input: {len(chunk.split())} words, {len(chunk)} chars")

        try:
            translated = engine.translate(chunk)
            translations.append(translated)

            chunk_time = time.time() - chunk_start
            print(f"   Output: {len(translated)} chars")
            print(f"   ✅ Completed in {chunk_time:.1f}s")

            # Progress
            elapsed = time.time() - start_time
            avg = elapsed / i
            remaining = len(chunks) - i
            eta = remaining * avg
            print(f"   📈 Progress: {i/len(chunks)*100:.1f}% | ETA: {eta/60:.1f}m")

        except Exception as e:
            print(f"   ❌ Error: {e}")
            translations.append(f"[TRANSLATION ERROR: {e}]")

    # Combine translations
    final_translation = "\n\n".join(translations)

    # Create output directory
    output_dir = Path("./translation_output")
    output_dir.mkdir(parents=True, exist_ok=True)

    # Save output
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = output_dir / f"translation_{target_code}_{timestamp}.txt"

    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(final_translation)

    # Summary
    total_time = time.time() - start_time
    trans_chars = len(final_translation)

    print(f"\n{'=' * 70}")
    print(f"🎉 TRANSLATION COMPLETE!")
    print(f"{'=' * 70}")
    print(f"⏱️ Time: {total_time/60:.1f} minutes")
    print(f"📦 Chunks: {len(chunks)}")
    print(f"⚡ Avg/chunk: {total_time/len(chunks):.1f}s")
    print(f"📝 Input: {orig_chars:,} chars")
    print(f"📝 Output: {trans_chars:,} chars")
    print(f"📊 Ratio: {trans_chars/orig_chars:.2f}x")
    print(f"💾 Output: {output_file}")
    print(f"{'=' * 70}")

    return str(output_file)


# Run translation
OUTPUT_FILE = translate_file(
    input_file=UPLOADED_FILE,
    source_lang=SOURCE_LANG_NAME,
    source_code=SOURCE_LANG_CODE,
    target_lang=TARGET_LANG_NAME,
    target_code=TARGET_LANG_CODE,
    chunk_size=CHUNK_SIZE
)

print(f"\n✅ Translation complete: {OUTPUT_FILE}")

## 📖 Step 7: Preview Translation

In [ ]:
from IPython.display import display, HTML
import os

if os.path.exists(OUTPUT_FILE):
    # Read and display translation
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
        translation = f.read()

    file_size = os.path.getsize(OUTPUT_FILE) / 1024  # KB
    word_count = len(translation.split())

    print(f"📊 Translation stats:")
    print(f"   Words: {word_count:,}")
    print(f"   Characters: {len(translation):,}")
    print(f"   File size: {file_size:.2f} KB")

    print(f"\n📖 Preview (first 1500 chars):")
    print(f"{'=' * 50}")
    print(translation[:1500])
    print(f"{'=' * 50}")
    if len(translation) > 1500:
        print(f"... [truncated, {len(translation) - 1500:,} more chars]")
else:
    print("❌ Output file not found. Please run the translation step again.")

## 📥 Step 8: Download Translation

In [ ]:
# Download the translated file
from google.colab import files

print("📥 Downloading your translated file...")
files.download(OUTPUT_FILE)
print("✅ Download started! Check your browser's download folder.")

## 💾 (Optional) Save to Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
import shutil

print("📂 Mounting Google Drive...")
drive.mount('/content/drive')

# Create output folder in Drive
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/TranslateGemma_Output"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

# Copy file to Drive
drive_output_path = os.path.join(DRIVE_OUTPUT_DIR, os.path.basename(OUTPUT_FILE))
shutil.copy(OUTPUT_FILE, drive_output_path)

print(f"\n✅ Translation saved to Google Drive:")
print(f"   📁 {drive_output_path}")

---

## 📚 TranslateGemma:27B Reference

### About the Model
TranslateGemma is Google's open translation model built on Gemma 3. The 27B parameter version offers the highest quality translations.

### Supported Languages (55)

| Region | Languages |
|--------|----------|
| **Indian** | Hindi, Bengali, Tamil, Telugu, Marathi, Gujarati, Kannada, Malayalam, Nepali |
| **European** | Spanish, French, German, Italian, Portuguese, Dutch, Polish, Romanian, Greek, Czech, Slovak, Hungarian, Bulgarian, Croatian, Serbian, Slovenian, Macedonian, Albanian, Norwegian, Swedish, Danish, Finnish, Estonian, Latvian, Lithuanian, Icelandic, Catalan |
| **Slavic** | Russian, Ukrainian, Belarusian |
| **Asian** | Chinese (Simplified), Japanese, Korean, Thai, Vietnamese, Indonesian, Filipino |
| **Middle Eastern** | Arabic, Persian, Hebrew, Turkish, Georgian, Azerbaijani, Kazakh, Mongolian |
| **African** | Swahili, Amharic, Afrikaans |

### Prompt Format
TranslateGemma uses a specific prompt structure for optimal results:
```
You are a professional {SOURCE_LANG} ({SOURCE_CODE}) to {TARGET_LANG} ({TARGET_CODE}) translator. 
Your goal is to accurately convey the meaning and nuances of the original {SOURCE_LANG} text 
while adhering to {TARGET_LANG} grammar, vocabulary, and cultural sensitivities. 
Produce only the {TARGET_LANG} translation, without any additional explanations or commentary. 
Please translate the following {SOURCE_LANG} text into {TARGET_LANG}: {TEXT}
```

### Tips
- Use GPU runtime (T4) for faster translations
- For long texts, chunk size of 200-300 words works best
- The model works best with clear, well-formatted input text
- Translation quality is highest for the 55 production-tier languages listed above